# OpenFF SPICE2 Subset Optimization Dataset v4.0

This notebook generates optimizations for suspicious SPICE2 records.

In [1]:
import pathlib
from pprint import pprint
import numpy as np
import pandas as pd

from openff.toolkit import Molecule
from openff.toolkit.utils import RDKitToolkitWrapper, ToolkitRegistry
from openff.qcsubmit import workflow_components
from openff.qcsubmit.factories import OptimizationDatasetFactory
from openff.qcsubmit.utils.visualize import molecules_to_pdf

In [2]:
smiles = (
    pd.read_csv("spice2_smiles.csv", usecols=[1], header=None, dtype=str)
    .iloc[:, 0]
    .dropna()
    .str.strip()
)
smiles = smiles[(smiles != "") & (smiles.str.lower() != "smiles")]

In [3]:
molecules = [
    Molecule.from_smiles(smi, allow_undefined_stereo=True)
    for smi in smiles
]
molecules_to_pdf(molecules, f"dataset.pdf")
len(molecules)

/var/folders/dr/9nnhm1493kv7_s9r_wj_l_jm0000gn/T/ipykernel_60603/2706337252.py:2: AtomMappingWarning: Warning! Fully mapped SMILES pattern passed to `from_smiles`. The atom map is stored as a property in `Molecule._properties`, but these indices are NOT used to determine atom ordering. To use these indices for atom ordering, use `Molecule.from_mapped_smiles`.
  Molecule.from_smiles(smi, allow_undefined_stereo=True)


629

In [4]:
dataset_factory = OptimizationDatasetFactory()
dataset_factory.add_workflow_components(
    workflow_components.StandardConformerGenerator(max_conformers=10)
)

description = """This optimization dataset targets molecules with suspicious conformers identified in the SPICE2 set and is intended
to improve geometry-level coverage for challenging charged chemistries. Molecules were ingested from 
`spice2_smiles.csv`, converted to OpenFF molecules with undefined stereochemistry allowed when needed, and expanded
with up to 10 conformers per molecule using the QCSubmit standard conformer generation workflow.

Computations use the OpenFF default optimization level of theory, B3LYP-D3BJ/DZVP (Psi4), with the standard OpenFF
optimization QC specification. The resulting set spans elements {P, S, C, Br, Cl, N, F, H, I, O}, includes formal
charges from -8.0 to +2.0, and covers a broad molecular-weight range (min 34.08, mean 288.23, max 1091.26 Da). This
submission is designed for force field training and validation workflows that require robust optimization data for
highly charged and chemically diverse SPICE2-derived molecules."""
toolkit_registry = ToolkitRegistry(toolkit_precedence=[RDKitToolkitWrapper()])

dataset = dataset_factory.create_dataset(
    dataset_name="OpenFF SPICE2 Subset Optimization Dataset v4.0",
    tagline="Improve optimization coverage for suspicious SPICE2 conformers across challenging charged chemistries",
    description=description,
    molecules=molecules,
    toolkit_registry=toolkit_registry,
)

dataset.metadata.submitter = "jaclark5"
dataset.metadata.long_description_url = (
    "https://github.com/openforcefield/qca-dataset-submission/tree/master/"
    "submissions/" + str(pathlib.Path.cwd().name)
)

Deduplication                 :   0%|                   | 0/629 [00:00<?, ?it/s]

StandardConformerGenerator    :   0%|                   | 0/628 [00:00<?, ?it/s][15:11:36] UFFTYPER: Warning: hybridization set to SP3 for atom 31
[15:11:36] UFFTYPER: Warning: hybridization set to SP3 for atom 34
[15:11:36] UFFTYPER: Warning: hybridization set to SP3 for atom 18
[15:11:36] UFFTYPER: Unrecognized atom type: S_5+4 (4)
[15:11:36] UFFTYPER: Warning: hybridization set to SP3 for atom 9
[15:11:36] UFFTYPER: Warning: hybridization set to SP3 for atom 19
StandardConformerGenerator    :   3%|▎         | 17/628 [00:06<02:39,  3.83it/s][15:11:37] UFFTYPER: Warning: hybridization set to SP3 for atom 30
[15:11:37] UFFTYPER: Unrecognized charge state for atom: 37
[15:11:37] UFFTYPER: Warning: hybridization set to SP3 for atom 11
StandardConformerGenerator    :   4%|▍         | 27/628 [00:06<01:37,  6.19it/s][15:11:37] UFFTYPER: Warning: hybridization set to SP3 for atom 11
[15:11:37] UFFTYPER: Warning: hybridization set to SP3 for atom 19
[15:11:37] UFFTYPER: Warning: hybridization

In [5]:
dataset.n_molecules

626

## QC Specifications

Add the default OpenFF spec and an OpenFF spec for charged and challenging molecules.

In [6]:
dataset.add_qc_spec(method='B3LYP-D3BJ',
                    basis='DZVP',
                    program='psi4',
                    spec_name="default",
                    spec_description="Standard OpenFF optimization quantum chemistry specification.",
                    store_wavefunction='none',
                    implicit_solvent=None,
                    maxiter=200,
                    scf_properties=['dipole', 'quadrupole', 'wiberg_lowdin_indices', 'mayer_indices', 'lowdin_charges', 'mulliken_charges'],
                    keywords={},
                    overwrite=True
                    )        


        

In [7]:
pprint(dataset.dict()['qc_specifications'])

{'default': {'basis': 'DZVP',
             'implicit_solvent': None,
             'keywords': {},
             'maxiter': 200,
             'method': 'B3LYP-D3BJ',
             'program': 'psi4',
             'scf_properties': ['dipole',
                                'quadrupole',
                                'wiberg_lowdin_indices',
                                'mayer_indices',
                                'lowdin_charges',
                                'mulliken_charges'],
             'spec_description': 'Standard OpenFF optimization quantum '
                                 'chemistry specification.',
             'spec_name': 'default',
             'store_wavefunction': 'none'}}


In [8]:
# summarize dataset for readme
confs = np.array([len(mol.conformers) for mol in dataset.molecules])

print("- Number of unique molecules:", dataset.n_molecules)
print("- Number of filtered molecules:", dataset.n_filtered)
print("- Number of conformers:", sum(confs))
print(
    "- Number of conformers per molecule (min, mean, max): "
    f"{confs.min()}, {confs.mean():.2f}, {confs.max()}"
)

masses = [
    [
        sum([atom.mass.m for atom in molecule.atoms])
        for molecule in dataset.molecules
    ]
]
print(f"- Mean molecular weight: {np.mean(np.array(masses)):.2f}")
print(f"- Min molecular weight: {np.min(np.array(masses)):.2f}")
print(f"- Max molecular weight: {np.max(np.array(masses)):.2f}")
print("- Charges:", sorted(set(m.total_charge.m for m in dataset.molecules)))


print("## Metadata")
print(f"- Elements: {{{', '.join(dataset.metadata.dict()['elements'])}}}")


fields = [
    "basis",
    "implicit_solvent",
    "keywords",
    "maxiter",
    "method",
    "program",
]
for spec, obj in dataset.qc_specifications.items():
    od = obj.dict()
    print("- Spec:", spec)
    for field in fields:
        print(f"\t - {field}: {od[field]}")
    print("\t- SCF properties:")
    for field in od["scf_properties"]:
        print(f"\t\t- {field}")


# export the dataset
dataset.export_dataset("dataset.json.bz2")
dataset.molecules_to_file("dataset.smi", "smi")
dataset.visualize("dataset.pdf", columns=4)

- Number of unique molecules: 626
- Number of filtered molecules: 2
- Number of conformers: 2350
- Number of conformers per molecule (min, mean, max): 1, 3.75, 10
- Mean molecular weight: 288.23
- Min molecular weight: 34.08
- Max molecular weight: 1091.26
- Charges: [-8.0, -4.0, -3.0, -2.0, -1.0, 0.0, 1.0, 2.0]
## Metadata
- Elements: {P, N, I, F, C, Cl, S, Br, H, O}
- Spec: default
	 - basis: DZVP
	 - implicit_solvent: None
	 - keywords: {}
	 - maxiter: 200
	 - method: B3LYP-D3BJ
	 - program: psi4
	- SCF properties:
		- dipole
		- quadrupole
		- wiberg_lowdin_indices
		- mayer_indices
		- lowdin_charges
		- mulliken_charges
